# YahooQuery Market Maker Volume Structure Research

Bu notebook, `yahooquery_gap_research_vectorbt.ipynb` dosyasının kopyası olarak oluşturuldu. Orijinal notebook değiştirilmedi. Veri çekme, BIST universe, ownership ve completeness filtreleri aynı kaldı; conventional gap yöntemi yerine market maker maliyetlenme, market structure, volume filter, absorption proxy ve micro-sweep yaklaşımı uygulanıyor.


## 1. Imports


In [ ]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import warnings

import numpy as np
import pandas as pd
import requests
from yahooquery import Ticker

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    tqdm = None

try:
    import nbformat
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Plotly notebook rendering requires nbformat. Run `pip install nbformat` "
        "inside the active devcontainer virtualenv."
    ) from exc

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import vectorbt as vbt
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "vectorbt is required for this notebook. Install it with `pip install vectorbt` "
        "or rebuild the devcontainer after the dependency update."
    ) from exc

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 160
pd.options.display.float_format = "{:.4f}".format


def log_step(title: str, **values) -> None:
    print(f"\n[{title}]")
    for key, value in values.items():
        print(f"  {key}: {value}")

def normalize_datetime_utc_naive(values) -> pd.Series:
    parsed = pd.to_datetime(values, utc=True, errors="coerce")
    if isinstance(parsed, pd.Series):
        return parsed.dt.tz_convert(None)
    return pd.Series(parsed).dt.tz_convert(None)



## 2. Research Parameters


In [ ]:
SYMBOLS = []
USE_BIST_UNIVERSE = True
BIST_UNIVERSE_LIMIT = None
PLOT_SYMBOL_LIMIT = 5
EXCLUDE_DOMINANT_HOLDER_SYMBOLS = True
DOMINANT_HOLDER_THRESHOLD = 0.50
OWNERSHIP_FILTER_VERBOSE = True
OWNERSHIP_PROGRESS_EVERY = 25
OWNERSHIP_MAX_WORKERS = 12
PARALLEL_MAX_WORKERS = 10
VISUALIZATION_TOP_N = 20

PERIOD = "60d"
INTERVAL = "5m"
VECTORBT_FREQ = "5min"
WEEKLY_PERIOD = "max"
WEEKLY_INTERVAL = "1wk"

INITIAL_CAPITAL = 100_000.0
MAX_RISK_FRACTION = 0.01
MAX_POSITION_RISK_TL = INITIAL_CAPITAL * MAX_RISK_FRACTION
TRAIN_FRACTION = 0.70
FEES = 0.001
SLIPPAGE = 0.001

PIVOT_LEFT = 3
PIVOT_RIGHT = 3
WEEKLY_PIVOT_LEFT = 2
WEEKLY_PIVOT_RIGHT = 2
MARKET_STRUCTURE_LOOKBACK_BARS = 1_500
ENTRY_LOOKAHEAD_BARS = 300

VOLUME_MA_WINDOW = 48
VOLUME_UPDATE_RATIO = 1.50
VOLUME_ENTRY_RATIO = 1.20
ABSORPTION_VOLUME_RATIO = 1.35
ABSORPTION_LOWER_WICK_RATIO = 0.40
MICRO_SWEEP_MAX_PCT = 0.006
MICRO_SWEEP_MIN_PCT = 0.0002
STOP_CLOSE_BUFFER_PCT = 0.0015
ATR_WINDOW = 14
STOP_ATR_BUFFER = 0.50
TAKE_PROFIT_R_MULTIPLIER = 2.50

PRICE_BUCKET_PCT = 0.0025
VALUE_AREA_RATIO = 0.70
COST_ZONE_PRICE_BUFFER_PCT = 0.004
COST_ZONE_MIN_VOLUME_SHARE = 0.18
VOLUME_PROFILE_DEMAND_DELTA_RATIO = 0.08
REQUIRE_LIQUIDITY_CONFIRMATION = True

log_step(
    "research_config",
    use_bist_universe=USE_BIST_UNIVERSE,
    bist_universe_limit=BIST_UNIVERSE_LIMIT,
    plot_symbol_limit=PLOT_SYMBOL_LIMIT,
    period=PERIOD,
    interval=INTERVAL,
    weekly_period=WEEKLY_PERIOD,
    weekly_interval=WEEKLY_INTERVAL,
    ownership_max_workers=OWNERSHIP_MAX_WORKERS,
    parallel_max_workers=PARALLEL_MAX_WORKERS,
    volume_update_ratio=VOLUME_UPDATE_RATIO,
    volume_entry_ratio=VOLUME_ENTRY_RATIO,
    micro_sweep_max_pct=MICRO_SWEEP_MAX_PCT,
    price_bucket_pct=PRICE_BUCKET_PCT,
    value_area_ratio=VALUE_AREA_RATIO,
    require_liquidity_confirmation=REQUIRE_LIQUIDITY_CONFIRMATION,
)



def run_parallel_tasks(items, worker, task_name: str, max_workers: int | None = None):
    items = list(items)
    if not items:
        return []
    workers = max(1, min(max_workers or PARALLEL_MAX_WORKERS, len(items)))
    started_at = time.perf_counter()
    if workers == 1 or len(items) == 1:
        iterator = items
        if tqdm is not None:
            iterator = tqdm(iterator, total=len(items), desc=task_name)
        results = [worker(item) for item in iterator]
    else:
        results = []
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = [executor.submit(worker, item) for item in items]
            iterator = as_completed(futures)
            if tqdm is not None:
                iterator = tqdm(iterator, total=len(futures), desc=task_name)
            for future in iterator:
                results.append(future.result())
    log_step(
        f"{task_name}_parallel_complete",
        tasks=len(items),
        workers=workers,
        elapsed_seconds=round(time.perf_counter() - started_at, 2),
    )
    return results


## 3. BIST Universe And Liquidity Filters


In [ ]:
def progress_write(message: str) -> None:
    if tqdm is not None:
        tqdm.write(message)
    else:
        print(message)


def get_bist_symbols():
    url = "https://scanner.tradingview.com/turkey/scan"
    payload = {
        "columns": ["name"],
        "filter": [
            {"left": "exchange", "operation": "equal", "right": "BIST"},
            {"left": "type", "operation": "equal", "right": "stock"},
        ],
        "range": [0, 1000],
    }
    started_at = time.perf_counter()
    print("[get_bist_symbols] requesting TradingView BIST scan...")
    response = requests.post(url, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
    symbols = [item["d"][0] for item in data["data"]]
    log_step(
        "tradingview_scan_complete",
        status_code=response.status_code,
        elapsed_seconds=round(time.perf_counter() - started_at, 2),
        raw_symbol_count=len(symbols),
        sample=symbols[:10],
    )
    return symbols


def to_yahoo_bist_symbol(symbol: str) -> str:
    symbol = str(symbol).strip().upper()
    if not symbol:
        return symbol
    return symbol if "." in symbol else f"{symbol}.IS"


def normalize_yahoo_symbols(symbols: list[str]) -> list[str]:
    return sorted({to_yahoo_bist_symbol(symbol) for symbol in symbols if str(symbol).strip()})


def payload_records(payload, symbol: str, source: str) -> list[dict]:
    if payload is None:
        return []
    if isinstance(payload, pd.DataFrame):
        frame = payload.reset_index()
        if "symbol" not in frame.columns:
            frame["symbol"] = symbol
        frame["source"] = source
        return frame.to_dict("records")
    if isinstance(payload, pd.Series):
        row = payload.to_dict()
        row.update({"symbol": symbol, "source": source})
        return [row]
    if isinstance(payload, list):
        rows = []
        for item in payload:
            rows.extend(payload_records(item, symbol, source))
        return rows
    if isinstance(payload, dict):
        if symbol in payload:
            return payload_records(payload[symbol], symbol, source)
        nested_rows = []
        for key, value in payload.items():
            if isinstance(value, (dict, list, pd.DataFrame, pd.Series)):
                nested_rows.extend(payload_records(value, symbol, f"{source}.{key}"))
        if nested_rows:
            return nested_rows
        row = dict(payload)
        row.update({"symbol": symbol, "source": source})
        return [row]
    return []


def coerce_holder_percent(value) -> float:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if isinstance(value, str):
        cleaned = value.strip().replace("%", "").replace(",", ".")
        if not cleaned:
            return np.nan
        value = cleaned
    try:
        pct = float(value)
    except (TypeError, ValueError):
        return np.nan
    return pct / 100.0 if pct > 1.0 else pct


def fetch_holder_snapshot(symbol: str, verbose: bool = False) -> pd.DataFrame:
    ticker = Ticker(symbol, asynchronous=False)
    rows = []
    source_timings = {}
    for source in ["institution_ownership", "fund_ownership", "major_holders"]:
        source_started_at = time.perf_counter()
        try:
            payload = getattr(ticker, source)
            source_rows = payload_records(payload, symbol, source)
            rows.extend(source_rows)
            source_timings[source] = {
                "rows": len(source_rows),
                "elapsed_s": round(time.perf_counter() - source_started_at, 2),
            }
        except Exception as exc:
            elapsed_s = round(time.perf_counter() - source_started_at, 2)
            source_timings[source] = {"rows": 0, "elapsed_s": elapsed_s, "error": str(exc)[:120]}
            rows.append({"symbol": symbol, "source": source, "holder_error": str(exc)})

    if verbose:
        progress_write(f"  [ownership_sources] {symbol} -> {source_timings}")

    if not rows:
        return pd.DataFrame(columns=["symbol", "source", "holder_name", "holder_pct"])

    frame = pd.DataFrame(rows)
    name_columns = ["organization", "holder", "Holder", "name", "Name"]
    percent_columns = ["pctHeld", "pct_held", "percentHeld", "Percent Held", "% Out", "% Shares Out", "sharesPercentSharesOut"]

    frame["holder_name"] = np.nan
    for column in name_columns:
        if column in frame.columns:
            frame["holder_name"] = frame["holder_name"].combine_first(frame[column])

    frame["holder_pct"] = np.nan
    for column in percent_columns:
        if column in frame.columns:
            frame["holder_pct"] = frame["holder_pct"].combine_first(frame[column].map(coerce_holder_percent))

    return frame


def evaluate_symbol_dominant_holder(
    index: int,
    symbol: str,
    threshold: float,
) -> dict:
    symbol_started_at = time.perf_counter()
    try:
        holder_rows = fetch_holder_snapshot(symbol, verbose=False)
        error_count = int(holder_rows["holder_error"].notna().sum()) if "holder_error" in holder_rows.columns else 0
        named_holders = holder_rows.dropna(subset=["holder_name", "holder_pct"]).copy()
        named_holders = named_holders[named_holders["holder_name"].astype(str).str.strip().ne("")]
        elapsed_s = round(time.perf_counter() - symbol_started_at, 2)

        if named_holders.empty:
            row = {
                "input_order": index,
                "symbol": symbol,
                "kept": True,
                "status": "kept_no_named_holder_data",
                "dominant_holder": None,
                "dominant_holder_pct": np.nan,
                "source": None,
                "raw_holder_rows": len(holder_rows),
                "named_holder_rows": 0,
                "elapsed_seconds": elapsed_s,
            }
            return {
                "row": row,
                "errors": error_count,
                "no_named_holder_data": 1,
                "status_message": f"kept(no holder data) elapsed={elapsed_s}s",
            }

        dominant = named_holders.sort_values("holder_pct", ascending=False).iloc[0]
        dominant_pct = float(dominant["holder_pct"])
        excluded = dominant_pct > threshold
        row = {
            "input_order": index,
            "symbol": symbol,
            "kept": not excluded,
            "status": "excluded_dominant_holder" if excluded else "kept",
            "dominant_holder": dominant["holder_name"],
            "dominant_holder_pct": dominant_pct,
            "source": dominant.get("source"),
            "raw_holder_rows": len(holder_rows),
            "named_holder_rows": len(named_holders),
            "elapsed_seconds": elapsed_s,
        }
        return {
            "row": row,
            "errors": error_count,
            "no_named_holder_data": 0,
            "status_message": (
                f"{'excluded' if excluded else 'kept'} "
                f"holder={dominant['holder_name']} pct={dominant_pct:.1%} elapsed={elapsed_s}s"
            ),
        }
    except Exception as exc:
        elapsed_s = round(time.perf_counter() - symbol_started_at, 2)
        row = {
            "input_order": index,
            "symbol": symbol,
            "kept": True,
            "status": "kept_holder_fetch_error",
            "dominant_holder": None,
            "dominant_holder_pct": np.nan,
            "source": None,
            "raw_holder_rows": 0,
            "named_holder_rows": 0,
            "elapsed_seconds": elapsed_s,
            "error": str(exc)[:300],
        }
        return {
            "row": row,
            "errors": 1,
            "no_named_holder_data": 1,
            "status_message": f"kept(fetch error) error={str(exc)[:120]} elapsed={elapsed_s}s",
        }


def filter_symbols_by_dominant_holder(
    symbols: list[str],
    threshold: float = DOMINANT_HOLDER_THRESHOLD,
    verbose: bool = OWNERSHIP_FILTER_VERBOSE,
    print_every: int = OWNERSHIP_PROGRESS_EVERY,
    max_workers: int = OWNERSHIP_MAX_WORKERS,
) -> tuple[list[str], pd.DataFrame]:
    report_rows = []
    counters = {"kept": 0, "excluded": 0, "no_named_holder_data": 0, "errors": 0}
    started_at = time.perf_counter()
    max_workers = max(1, min(int(max_workers), len(symbols) or 1))

    log_step(
        "ownership_filter_start",
        symbol_count=len(symbols),
        threshold=threshold,
        verbose=verbose,
        print_every=print_every,
        max_workers=max_workers,
    )

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(evaluate_symbol_dominant_holder, index, symbol, threshold)
            for index, symbol in enumerate(symbols, start=1)
        ]
        completed_futures = as_completed(futures)
        if tqdm is not None:
            completed_futures = tqdm(
                completed_futures,
                total=len(futures),
                desc=f"ownership filter x{max_workers}",
                unit="symbol",
            )

        for completed_count, future in enumerate(completed_futures, start=1):
            result = future.result()
            row = result["row"]
            report_rows.append(row)

            counters["kept"] += int(bool(row["kept"]))
            counters["excluded"] += int(row["status"] == "excluded_dominant_holder")
            counters["no_named_holder_data"] += int(result["no_named_holder_data"])
            counters["errors"] += int(result["errors"])

            if tqdm is not None and hasattr(completed_futures, "set_postfix"):
                completed_futures.set_postfix(
                    kept=counters["kept"],
                    excluded=counters["excluded"],
                    no_data=counters["no_named_holder_data"],
                    errors=counters["errors"],
                )

            should_print = verbose and (
                completed_count == 1
                or completed_count == len(symbols)
                or completed_count % max(1, print_every) == 0
                or row["status"] in {"excluded_dominant_holder", "kept_holder_fetch_error"}
            )
            if should_print:
                progress_write(
                    f"  [{completed_count}/{len(symbols)} done] {row['symbol']}: {result['status_message']} | "
                    f"kept={counters['kept']} excluded={counters['excluded']} no_data={counters['no_named_holder_data']} errors={counters['errors']}"
                )

    report = pd.DataFrame(report_rows).sort_values("input_order").reset_index(drop=True)
    kept_symbols = report.loc[report["kept"], "symbol"].to_list()

    log_step(
        "ownership_filter_complete",
        elapsed_seconds=round(time.perf_counter() - started_at, 2),
        max_workers=max_workers,
        kept_symbol_count=len(kept_symbols),
        excluded_symbol_count=counters["excluded"],
        no_holder_data_count=counters["no_named_holder_data"],
        error_count=counters["errors"],
    )
    return kept_symbols, report


log_step(
    "symbol_universe_start",
    use_bist_universe=USE_BIST_UNIVERSE,
    input_symbols=SYMBOLS if SYMBOLS else "from_tradingview",
)

if USE_BIST_UNIVERSE:
    raw_bist_symbols = get_bist_symbols()
    log_step("tradingview_bist_scan", raw_symbol_count=len(raw_bist_symbols), sample=raw_bist_symbols[:10])
    SYMBOLS = normalize_yahoo_symbols(raw_bist_symbols)
    if BIST_UNIVERSE_LIMIT is not None:
        SYMBOLS = SYMBOLS[: int(BIST_UNIVERSE_LIMIT)]
        print(f"  applied BIST_UNIVERSE_LIMIT -> {len(SYMBOLS)} symbols")
else:
    SYMBOLS = normalize_yahoo_symbols(SYMBOLS)

log_step("normalized_yahoo_symbols", symbol_count=len(SYMBOLS), sample=SYMBOLS[:10])

if EXCLUDE_DOMINANT_HOLDER_SYMBOLS:
    SYMBOLS, holder_report = filter_symbols_by_dominant_holder(SYMBOLS)
else:
    holder_report = pd.DataFrame({"symbol": SYMBOLS, "status": "ownership_filter_disabled"})

display(holder_report)
if "status" in holder_report.columns:
    status_counts = holder_report["status"].value_counts().rename_axis("status").reset_index(name="count")
    px.bar(
        status_counts,
        x="status",
        y="count",
        color="status",
        title="Ownership Filter Result Counts",
    ).show()
if "dominant_holder_pct" in holder_report.columns:
    holder_pct = holder_report.dropna(subset=["dominant_holder_pct"]).copy()
    if not holder_pct.empty:
        px.histogram(
            holder_pct,
            x="dominant_holder_pct",
            color="status" if "status" in holder_pct.columns else None,
            nbins=40,
            title="Dominant Holder Percent Distribution",
            labels={"dominant_holder_pct": "Dominant holder share"},
        ).show()
        top_holders = holder_pct.sort_values("dominant_holder_pct", ascending=False).head(VISUALIZATION_TOP_N)
        px.bar(
            top_holders,
            x="symbol",
            y="dominant_holder_pct",
            color="status" if "status" in top_holders.columns else None,
            hover_data=[column for column in ["dominant_holder", "source"] if column in top_holders.columns],
            title="Highest Dominant Holder Shares",
        ).show()
log_step(
    "ownership_filter_result",
    kept_symbol_count=len(SYMBOLS),
    excluded_symbol_count=int(holder_report["status"].eq("excluded_dominant_holder").sum()) if "status" in holder_report.columns else 0,
    kept_sample=SYMBOLS[:10],
)
print(f"Selected symbols ({len(SYMBOLS)}): {SYMBOLS}")
if not SYMBOLS:
    raise ValueError("No symbols left after BIST universe / ownership filters.")


## 4. Pull 5-Minute Data From YahooQuery


In [ ]:
log_step("yahoo_request", symbol_count=len(SYMBOLS), period=PERIOD, interval=INTERVAL, sample=SYMBOLS[:10])
raw_history = Ticker(SYMBOLS, asynchronous=False).history(period=PERIOD, interval=INTERVAL)

if raw_history is None or len(raw_history) == 0:
    raise RuntimeError("YahooQuery returned no rows for the requested symbols.")

prices_pd = raw_history.reset_index()
prices_pd = prices_pd.rename(columns={"level_0": "symbol"})
if "date" not in prices_pd.columns and "index" in prices_pd.columns:
    prices_pd = prices_pd.rename(columns={"index": "date"})

required = ["symbol", "date", "open", "high", "low", "close", "volume"]
missing = [column for column in required if column not in prices_pd.columns]
if missing:
    raise ValueError(f"YahooQuery output is missing columns: {missing}. Columns={list(prices_pd.columns)}")

prices_pd = prices_pd[required].copy()
prices_pd["date"] = normalize_datetime_utc_naive(prices_pd["date"])
log_step(
    "yahoo_raw_result",
    rows=len(prices_pd),
    returned_symbol_count=prices_pd["symbol"].nunique(),
    min_date=prices_pd["date"].min(),
    max_date=prices_pd["date"].max(),
)

available_symbols = sorted(prices_pd["symbol"].dropna().unique())
missing_symbols = sorted(set(SYMBOLS).difference(available_symbols))
if missing_symbols:
    print(f"YahooQuery returned no rows for these symbols and they will be removed: {missing_symbols}")

completeness_report = (
    prices_pd.groupby("symbol", observed=True)
    .agg(
        rows=("date", "size"),
        duplicate_timestamps=("date", lambda values: int(values.duplicated().sum())),
        **{f"{column}_missing": (column, lambda values: int(values.isna().sum())) for column in required},
    )
    .reset_index()
)
missing_columns = [column for column in completeness_report.columns if column.endswith("_missing")]
incomplete_mask = completeness_report[missing_columns].sum(axis=1).gt(0) | completeness_report["duplicate_timestamps"].gt(0)
incomplete_symbols = completeness_report.loc[incomplete_mask, "symbol"].to_list()

display(completeness_report)
log_step(
    "completeness_check",
    checked_symbols=len(completeness_report),
    incomplete_symbol_count=len(incomplete_symbols),
    incomplete_symbols=incomplete_symbols[:20],
)
if incomplete_symbols:
    print(f"Symbols removed because OHLCV/date columns are incomplete or duplicated: {incomplete_symbols}")
    prices_pd = prices_pd[~prices_pd["symbol"].isin(incomplete_symbols)].copy()

prices_pd = prices_pd.dropna(subset=required).sort_values(["symbol", "date"]).reset_index(drop=True)
SYMBOLS = sorted(prices_pd["symbol"].unique())
if not SYMBOLS:
    raise ValueError("No symbols left after YahooQuery completeness validation.")

summary = (
    prices_pd.groupby("symbol")
    .agg(
        rows=("date", "size"),
        min_date=("date", "min"),
        max_date=("date", "max"),
        last_close=("close", "last"),
        avg_volume=("volume", "mean"),
        avg_notional=("volume", lambda values: np.nan),
    )
    .reset_index()
)
notional = prices_pd.assign(notional=prices_pd["close"] * prices_pd["volume"])
notional_summary = notional.groupby("symbol", observed=True)["notional"].mean().rename("avg_notional").reset_index()
summary = summary.drop(columns=["avg_notional"]).merge(notional_summary, on="symbol", how="left")
plot_symbol_report = summary.sort_values(["rows", "avg_notional", "avg_volume"], ascending=[False, False, False]).head(PLOT_SYMBOL_LIMIT)
PLOT_SYMBOLS = plot_symbol_report["symbol"].to_list()

display(summary)
liquidity_plot = summary.sort_values("avg_notional", ascending=False).head(VISUALIZATION_TOP_N)
px.bar(
    liquidity_plot,
    x="symbol",
    y="avg_notional",
    color="rows",
    title="Top Symbols By Average 5m Notional Volume",
).show()
px.scatter(
    summary,
    x="rows",
    y="avg_notional",
    size="avg_volume",
    hover_name="symbol",
    title="Data Completeness vs Liquidity",
).show()
for symbol in PLOT_SYMBOLS:
    sample = prices_pd[prices_pd["symbol"] == symbol].tail(400).copy()
    if sample.empty:
        continue
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.03)
    fig.add_trace(
        go.Candlestick(x=sample["date"], open=sample["open"], high=sample["high"], low=sample["low"], close=sample["close"], name=symbol),
        row=1,
        col=1,
    )
    fig.add_trace(go.Bar(x=sample["date"], y=sample["volume"], name="Volume", marker_color="#64748b"), row=2, col=1)
    fig.update_layout(title=f"{symbol} Raw 5m Price And Volume", xaxis_rangeslider_visible=False, height=720)
    fig.show()
log_step(
    "validated_price_data",
    final_symbol_count=len(SYMBOLS),
    final_rows=len(prices_pd),
    selected_plot_symbols=PLOT_SYMBOLS,
)
prices_pd.tail()


## 5. Market Maker Structure + Volume Method

Bu kopya notebook'ta conventional gap yöntemi yok. Aynı BIST/universe/ownership/completeness filtrelerinden geçen 5 dakikalık veriyle şu yöntemi deniyoruz:

1. Haftalık grafikte makro yapı okunur: son majör tepe, o tepeden sonraki dip ve o dipten sonraki rebound tepe çıkarılır.
2. 5 dakikalık grafikte son satış dalgasının global dip bölgesi bulunur.
3. Global dipten sonra gelen ilk güçlü tepenin önceki tepeyi yukarı kırıp kırmadığı kontrol edilir. Bu CHoCH / update kabul edilir.
4. Update sonrası gelen dip, global dipten yüksek kalıyorsa higher-low / maliyet savunması kabul edilir.
5. Gerçek kademe verisi Yahoo'da olmadığı için order-book yerine iki proxy kullanıyoruz: bar bazlı volume/wick absorption ve discrete price bucket volume profile.
6. Volume profile'da fiyatlar bucket'lanır; her bucket için tahmini buy volume, sell volume, toplam volume, delta, POC ve value area hesaplanır.
7. Long entry, higher-low sonrası gelen momentum/reclaim barında veya hafif likidite süpürüp tekrar yapının içine dönüşte üretilir.
8. Stop mantığı wick altına değil, gövde kapanışıyla yapı bozulmasına dayanır: close global dip buffer altına inerse çıkılır.

Not: Bu gerçek Level-2/kademe analizi değildir; Yahoo OHLCV ile kademe davranışını yaklaşık temsil eden bir research proxy'dir. `buy_volume_proxy` ve `sell_volume_proxy`, barın kapanış konumundan türetilir. Gerçek order book / trade tape gelirse bu kısım doğrudan gerçek agresif alış-satış hacmiyle değiştirilebilir.


## 6. Weekly Macro Structure


In [ ]:
log_step("weekly_macro_request", symbol_count=len(SYMBOLS), period=WEEKLY_PERIOD, interval=WEEKLY_INTERVAL)
weekly_raw = Ticker(SYMBOLS, asynchronous=False).history(period=WEEKLY_PERIOD, interval=WEEKLY_INTERVAL)

if weekly_raw is None or len(weekly_raw) == 0:
    raise RuntimeError("YahooQuery returned no weekly rows for selected symbols.")

weekly_pd = weekly_raw.reset_index().rename(columns={"level_0": "symbol"})
if "date" not in weekly_pd.columns and "index" in weekly_pd.columns:
    weekly_pd = weekly_pd.rename(columns={"index": "date"})
weekly_pd = weekly_pd[["symbol", "date", "open", "high", "low", "close", "volume"]].copy()
weekly_pd["date"] = normalize_datetime_utc_naive(weekly_pd["date"])
weekly_pd = weekly_pd.dropna(subset=["symbol", "date", "open", "high", "low", "close"]).sort_values(["symbol", "date"])

def add_pivots(frame: pd.DataFrame, left: int, right: int) -> pd.DataFrame:
    frame = frame.sort_values("date").reset_index(drop=True).copy()
    highs = frame["high"].to_numpy(dtype=float)
    lows = frame["low"].to_numpy(dtype=float)
    pivot_high = np.zeros(len(frame), dtype=bool)
    pivot_low = np.zeros(len(frame), dtype=bool)
    for idx in range(left, max(left, len(frame) - right)):
        high_window = highs[idx - left : idx + right + 1]
        low_window = lows[idx - left : idx + right + 1]
        pivot_high[idx] = np.isfinite(highs[idx]) and highs[idx] == np.nanmax(high_window)
        pivot_low[idx] = np.isfinite(lows[idx]) and lows[idx] == np.nanmin(low_window)
    frame["pivot_high"] = pivot_high
    frame["pivot_low"] = pivot_low
    frame["pivot_high_confirmed_at"] = frame["date"].shift(-right)
    frame["pivot_low_confirmed_at"] = frame["date"].shift(-right)
    return frame


def weekly_macro_structure(symbol_frame: pd.DataFrame) -> dict:
    frame = add_pivots(symbol_frame, WEEKLY_PIVOT_LEFT, WEEKLY_PIVOT_RIGHT)
    highs = frame[frame["pivot_high"]].copy()
    if highs.empty:
        high_idx = int(frame["high"].idxmax())
    else:
        high_idx = int(highs.index[-1])

    after_high = frame.loc[high_idx + 1 :]
    if after_high.empty:
        dip_idx = int(frame["low"].idxmin())
    else:
        dip_idx = int(after_high["low"].idxmin())

    after_dip = frame.loc[dip_idx + 1 :]
    rebound_idx = int(after_dip["high"].idxmax()) if not after_dip.empty else dip_idx

    last_major_high = float(frame.loc[high_idx, "high"])
    post_high_dip = float(frame.loc[dip_idx, "low"])
    rebound_high = float(frame.loc[rebound_idx, "high"])
    macro_update = rebound_high > last_major_high

    return {
        "symbol": frame.loc[0, "symbol"],
        "weekly_last_major_high_time": frame.loc[high_idx, "date"],
        "weekly_last_major_high": last_major_high,
        "weekly_post_high_dip_time": frame.loc[dip_idx, "date"],
        "weekly_post_high_dip": post_high_dip,
        "weekly_rebound_high_time": frame.loc[rebound_idx, "date"],
        "weekly_rebound_high": rebound_high,
        "weekly_rebound_pct_from_dip": rebound_high / post_high_dip - 1.0 if post_high_dip else np.nan,
        "weekly_macro_update": bool(macro_update),
        "weekly_rows": len(frame),
    }


def weekly_macro_structure_task(item):
    _, frame = item
    return weekly_macro_structure(frame)

weekly_groups = [(symbol, frame.copy()) for symbol, frame in weekly_pd.groupby("symbol", observed=True)]
weekly_structure = pd.DataFrame(run_parallel_tasks(weekly_groups, weekly_macro_structure_task, "weekly_macro_structure"))
weekly_structure = weekly_structure.sort_values(["weekly_macro_update", "weekly_rebound_pct_from_dip"], ascending=[False, False]).reset_index(drop=True)
display(weekly_structure)
log_step(
    "weekly_macro_structure_ready",
    symbols=len(weekly_structure),
    macro_update_count=int(weekly_structure["weekly_macro_update"].sum()),
)

weekly_plot = weekly_structure.head(VISUALIZATION_TOP_N).copy()
px.bar(
    weekly_plot,
    x="symbol",
    y="weekly_rebound_pct_from_dip",
    color="weekly_macro_update",
    title="Weekly Macro Structure: Rebound From Post-High Dip",
    labels={"weekly_rebound_pct_from_dip": "Rebound From Dip", "symbol": "Symbol"},
).show()

for symbol in PLOT_SYMBOLS:
    sample = weekly_pd[weekly_pd["symbol"] == symbol].copy()
    if sample.empty:
        continue
    structure_row = weekly_structure[weekly_structure["symbol"] == symbol]
    fig = go.Figure()
    fig.add_trace(go.Candlestick(x=sample["date"], open=sample["open"], high=sample["high"], low=sample["low"], close=sample["close"], name=symbol))
    if not structure_row.empty:
        row = structure_row.iloc[0]
        fig.add_hline(y=row["weekly_last_major_high"], line_dash="dash", line_color="#dc2626", annotation_text="major high")
        fig.add_hline(y=row["weekly_post_high_dip"], line_dash="dot", line_color="#7c2d12", annotation_text="post-high dip")
        fig.add_hline(y=row["weekly_rebound_high"], line_dash="dash", line_color="#16a34a", annotation_text="rebound high")
    fig.update_layout(title=f"{symbol} Weekly Macro Structure", xaxis_rangeslider_visible=False, height=620)
    fig.show()


## 7. Intraday Volume And Pivot Features


In [ ]:
def build_intraday_volume_features_for_symbol(item) -> pd.DataFrame:
    symbol, symbol_frame = item
    symbol_frame = symbol_frame.sort_values("date").reset_index(drop=True).copy()
    symbol_frame["prev_close"] = symbol_frame["close"].shift(1)
    symbol_frame["prev_high"] = symbol_frame["high"].shift(1)
    symbol_frame["prev_low"] = symbol_frame["low"].shift(1)
    true_range = pd.concat(
        [
            (symbol_frame["high"] - symbol_frame["low"]).abs(),
            (symbol_frame["high"] - symbol_frame["prev_close"]).abs(),
            (symbol_frame["low"] - symbol_frame["prev_close"]).abs(),
        ],
        axis=1,
    ).max(axis=1)
    symbol_frame["true_range"] = true_range
    symbol_frame["atr"] = true_range.rolling(ATR_WINDOW, min_periods=3).mean()
    symbol_frame["volume_ma"] = symbol_frame["volume"].rolling(VOLUME_MA_WINDOW, min_periods=8).mean()
    symbol_frame["volume_ratio"] = symbol_frame["volume"] / symbol_frame["volume_ma"].replace(0, np.nan)
    candle_range = (symbol_frame["high"] - symbol_frame["low"]).replace(0, np.nan)
    symbol_frame["body_ratio"] = (symbol_frame["close"] - symbol_frame["open"]).abs() / candle_range
    symbol_frame["lower_wick_ratio"] = (symbol_frame[["open", "close"]].min(axis=1) - symbol_frame["low"]) / candle_range
    symbol_frame["upper_wick_ratio"] = (symbol_frame["high"] - symbol_frame[["open", "close"]].max(axis=1)) / candle_range
    symbol_frame["bullish_bar"] = symbol_frame["close"] > symbol_frame["open"]
    symbol_frame["bearish_bar"] = symbol_frame["close"] < symbol_frame["open"]
    symbol_frame = add_pivots(symbol_frame, PIVOT_LEFT, PIVOT_RIGHT)
    return symbol_frame


def add_intraday_volume_features(frame: pd.DataFrame) -> pd.DataFrame:
    groups = [(symbol, symbol_frame.copy()) for symbol, symbol_frame in frame.sort_values(["symbol", "date"]).groupby("symbol", observed=True)]
    frames = run_parallel_tasks(groups, build_intraday_volume_features_for_symbol, "intraday_volume_features")
    return pd.concat(frames, ignore_index=True).sort_values(["symbol", "date"]).reset_index(drop=True)

intraday = add_intraday_volume_features(prices_pd)
intraday = intraday.merge(weekly_structure, on="symbol", how="left")

feature_summary = intraday.groupby("symbol", observed=True).agg(
    rows=("date", "size"),
    pivot_highs=("pivot_high", "sum"),
    pivot_lows=("pivot_low", "sum"),
    avg_volume_ratio=("volume_ratio", "mean"),
    max_volume_ratio=("volume_ratio", "max"),
    high_volume_bars=("volume_ratio", lambda values: int((values >= VOLUME_UPDATE_RATIO).sum())),
    avg_atr=("atr", "mean"),
).reset_index()
display(feature_summary.sort_values("high_volume_bars", ascending=False))
log_step(
    "intraday_features_ready",
    rows=len(intraday),
    symbols=intraday["symbol"].nunique(),
    high_volume_bars=int((intraday["volume_ratio"] >= VOLUME_UPDATE_RATIO).sum()),
)

feature_plot = feature_summary.sort_values("high_volume_bars", ascending=False).head(VISUALIZATION_TOP_N)
px.bar(
    feature_plot,
    x="symbol",
    y="high_volume_bars",
    color="avg_volume_ratio",
    title="High-Volume 5m Bars By Symbol",
    labels={"high_volume_bars": "Bars Above Volume Threshold", "avg_volume_ratio": "Avg Volume Ratio"},
).show()

px.scatter(
    feature_summary,
    x="pivot_lows",
    y="pivot_highs",
    size="high_volume_bars",
    color="avg_volume_ratio",
    hover_name="symbol",
    title="Pivot Density vs Volume Pressure",
).show()

volume_ratio_sample = intraday[intraday["symbol"].isin(PLOT_SYMBOLS)].copy()
px.histogram(
    volume_ratio_sample,
    x="volume_ratio",
    color="symbol",
    nbins=80,
    barmode="overlay",
    title="Volume Ratio Distribution For Plotted Symbols",
).show()


## 8. Discrete Price Bucket Volume Profile Proxy

Kademe verisi olmadığı için fiyatı discrete bucket'lara bölüyoruz. Her bucket için şunları hesaplıyoruz:

- `buy_volume_proxy`: kapanış bar range'inin üst tarafına yakınsa alış baskısı yüksek varsayılır.
- `sell_volume_proxy`: kapanış bar range'inin alt tarafına yakınsa satış baskısı yüksek varsayılır.
- `vp_total_volume`: bucket'taki toplam hacim.
- `vp_delta`: buy proxy eksi sell proxy.
- `vp_delta_ratio`: bucket içindeki net baskı oranı.
- `vp_poc_price`: en yüksek hacimli fiyat bucket'ı.
- `vp_value_area`: hacmin yaklaşık yüzde 70'ini taşıyan bucket alanı.

Bu, gerçek kademe değil ama maliyet bölgesi / market maker savunması için OHLCV ile kurulabilecek en anlamlı proxy'lerden biri.


In [ ]:
def infer_buy_sell_volume_proxy(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    candle_range = (output["high"] - output["low"]).replace(0, np.nan)
    close_location = ((output["close"] - output["low"]) / candle_range).clip(0.0, 1.0)
    directional_location = pd.Series(
        np.where(output["close"] > output["open"], 0.65, np.where(output["close"] < output["open"], 0.35, 0.50)),
        index=output.index,
    )
    close_location = close_location.fillna(directional_location)
    output["typical_price"] = (output["high"] + output["low"] + output["close"]) / 3.0
    output["buy_volume_proxy"] = output["volume"] * close_location
    output["sell_volume_proxy"] = output["volume"] * (1.0 - close_location)
    output["volume_delta_proxy"] = output["buy_volume_proxy"] - output["sell_volume_proxy"]
    return output


def symbol_price_bucket_size(symbol_frame: pd.DataFrame) -> float:
    median_close = float(symbol_frame["close"].median())
    median_range = float((symbol_frame["high"] - symbol_frame["low"]).replace(0, np.nan).median())
    pct_bucket = median_close * PRICE_BUCKET_PCT if np.isfinite(median_close) and median_close > 0 else np.nan
    range_bucket = median_range / 2.0 if np.isfinite(median_range) and median_range > 0 else np.nan
    candidates = [value for value in [pct_bucket, range_bucket] if np.isfinite(value) and value > 0]
    return max(min(candidates), 0.001) if candidates else 0.01


def build_volume_profile(symbol_frame: pd.DataFrame) -> pd.DataFrame:
    if symbol_frame.empty:
        return pd.DataFrame(
            columns=[
                "symbol",
                "price_bin_mid",
                "vp_total_volume",
                "vp_buy_volume",
                "vp_sell_volume",
                "vp_delta",
                "vp_total_volume_share",
                "vp_delta_ratio",
                "vp_value_area",
                "vp_poc_price",
            ]
        )

    frame = infer_buy_sell_volume_proxy(symbol_frame)
    bucket_size = symbol_price_bucket_size(frame)
    frame["price_bin_mid"] = (frame["typical_price"] / bucket_size).round() * bucket_size
    profile = (
        frame.groupby(["symbol", "price_bin_mid"], observed=True)
        .agg(
            vp_total_volume=("volume", "sum"),
            vp_buy_volume=("buy_volume_proxy", "sum"),
            vp_sell_volume=("sell_volume_proxy", "sum"),
            bars=("date", "size"),
        )
        .reset_index()
        .sort_values(["symbol", "price_bin_mid"])
    )
    profile["vp_delta"] = profile["vp_buy_volume"] - profile["vp_sell_volume"]
    total_volume = float(profile["vp_total_volume"].sum())
    profile["vp_total_volume_share"] = profile["vp_total_volume"] / total_volume if total_volume > 0 else np.nan
    profile["vp_delta_ratio"] = profile["vp_delta"] / profile["vp_total_volume"].replace(0, np.nan)

    if profile.empty:
        profile["vp_value_area"] = False
        profile["vp_poc_price"] = np.nan
        return profile

    poc_row = profile.loc[profile["vp_total_volume"].idxmax()]
    poc_price = float(poc_row["price_bin_mid"])
    ordered = profile.sort_values("vp_total_volume", ascending=False).copy()
    ordered["cumulative_share"] = ordered["vp_total_volume"].cumsum() / total_volume if total_volume > 0 else np.nan
    value_bins = ordered.loc[ordered["cumulative_share"] <= VALUE_AREA_RATIO, "price_bin_mid"].to_list()
    crossing = ordered.loc[ordered["cumulative_share"] > VALUE_AREA_RATIO, "price_bin_mid"]
    if not crossing.empty:
        value_bins.append(float(crossing.iloc[0]))
    if not value_bins:
        value_bins = [poc_price]
    profile["vp_value_area"] = profile["price_bin_mid"].isin(value_bins)
    profile["vp_poc_price"] = poc_price
    return profile


def attach_volume_profile_proxy_for_symbol(item) -> tuple[pd.DataFrame, pd.DataFrame]:
    symbol, symbol_frame = item
    symbol_frame = infer_buy_sell_volume_proxy(symbol_frame.sort_values("date").reset_index(drop=True))
    bucket_size = symbol_price_bucket_size(symbol_frame)
    symbol_frame["price_bucket_size"] = bucket_size
    symbol_frame["price_bin_mid"] = (symbol_frame["typical_price"] / bucket_size).round() * bucket_size
    profile = build_volume_profile(symbol_frame)
    profile["vp_is_poc"] = np.isclose(profile["price_bin_mid"], profile["vp_poc_price"])
    symbol_frame = symbol_frame.merge(
        profile[
            [
                "symbol",
                "price_bin_mid",
                "vp_total_volume",
                "vp_buy_volume",
                "vp_sell_volume",
                "vp_delta",
                "vp_total_volume_share",
                "vp_delta_ratio",
                "vp_value_area",
                "vp_poc_price",
                "vp_is_poc",
            ]
        ],
        on=["symbol", "price_bin_mid"],
        how="left",
    )
    return symbol_frame, profile


def attach_volume_profile_proxy(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    groups = [(symbol, symbol_frame.copy()) for symbol, symbol_frame in frame.groupby("symbol", observed=True)]
    results = run_parallel_tasks(groups, attach_volume_profile_proxy_for_symbol, "volume_profile_proxy")
    enriched_frames = [result[0] for result in results]
    profiles = [result[1] for result in results]
    return (
        pd.concat(enriched_frames, ignore_index=True).sort_values(["symbol", "date"]).reset_index(drop=True),
        pd.concat(profiles, ignore_index=True).sort_values(["symbol", "price_bin_mid"]).reset_index(drop=True),
    )


def setup_volume_profile_confirmation(
    setup_window: pd.DataFrame,
    global_dip_price: float,
    higher_low_price: float,
) -> dict:
    profile = build_volume_profile(setup_window)
    if profile.empty:
        return {
            "profile_poc_price": np.nan,
            "profile_value_area_low": np.nan,
            "profile_value_area_high": np.nan,
            "cost_zone_volume_share": np.nan,
            "cost_zone_delta_ratio": np.nan,
            "volume_profile_confirmed": False,
        }

    zone_low = min(global_dip_price, higher_low_price) * (1.0 - COST_ZONE_PRICE_BUFFER_PCT)
    zone_high = max(global_dip_price, higher_low_price) * (1.0 + COST_ZONE_PRICE_BUFFER_PCT)
    zone_profile = profile[profile["price_bin_mid"].between(zone_low, zone_high)].copy()
    total_volume = float(profile["vp_total_volume"].sum())
    zone_volume = float(zone_profile["vp_total_volume"].sum()) if not zone_profile.empty else 0.0
    zone_delta = float(zone_profile["vp_delta"].sum()) if not zone_profile.empty else 0.0
    zone_share = zone_volume / total_volume if total_volume > 0 else np.nan
    zone_delta_ratio = zone_delta / zone_volume if zone_volume > 0 else np.nan
    value_area = profile[profile["vp_value_area"]]
    profile_poc_price = float(profile["vp_poc_price"].iloc[0])
    profile_value_area_low = float(value_area["price_bin_mid"].min()) if not value_area.empty else np.nan
    profile_value_area_high = float(value_area["price_bin_mid"].max()) if not value_area.empty else np.nan
    volume_profile_confirmed = bool(
        zone_volume > 0
        and (
            (pd.notna(zone_share) and zone_share >= COST_ZONE_MIN_VOLUME_SHARE and (pd.isna(zone_delta_ratio) or zone_delta_ratio >= 0.0))
            or (pd.notna(zone_delta_ratio) and zone_delta_ratio >= VOLUME_PROFILE_DEMAND_DELTA_RATIO)
        )
    )
    return {
        "profile_poc_price": profile_poc_price,
        "profile_value_area_low": profile_value_area_low,
        "profile_value_area_high": profile_value_area_high,
        "cost_zone_volume_share": zone_share,
        "cost_zone_delta_ratio": zone_delta_ratio,
        "volume_profile_confirmed": volume_profile_confirmed,
    }


intraday, volume_profile = attach_volume_profile_proxy(intraday)
profile_summary = (
    volume_profile.sort_values(["symbol", "vp_total_volume"], ascending=[True, False])
    .groupby("symbol", observed=True)
    .head(5)
    .reset_index(drop=True)
)
display(profile_summary)
volume_profile_symbol_summary = volume_profile.groupby("symbol", observed=True).agg(
    price_buckets=("price_bin_mid", "size"),
    total_volume=("vp_total_volume", "sum"),
    poc_price=("vp_poc_price", "first"),
    positive_delta_buckets=("vp_delta_ratio", lambda values: int((values > 0).sum())),
    negative_delta_buckets=("vp_delta_ratio", lambda values: int((values < 0).sum())),
    value_area_buckets=("vp_value_area", "sum"),
).reset_index()
display(volume_profile_symbol_summary.sort_values("total_volume", ascending=False).head(VISUALIZATION_TOP_N))
log_step(
    "volume_profile_proxy_ready",
    symbols=volume_profile["symbol"].nunique(),
    price_buckets=len(volume_profile),
    value_area_buckets=int(volume_profile["vp_value_area"].sum()),
)

px.bar(
    volume_profile_symbol_summary.sort_values("total_volume", ascending=False).head(VISUALIZATION_TOP_N),
    x="symbol",
    y="total_volume",
    color="value_area_buckets",
    title="Volume Profile Total Volume By Symbol",
).show()

px.scatter(
    volume_profile_symbol_summary,
    x="price_buckets",
    y="positive_delta_buckets",
    size="total_volume",
    color="negative_delta_buckets",
    hover_name="symbol",
    title="Demand/Supply Bucket Balance By Symbol",
).show()

for symbol in PLOT_SYMBOLS:
    profile = volume_profile[volume_profile["symbol"] == symbol].sort_values("price_bin_mid").copy()
    if profile.empty:
        continue
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=profile["vp_buy_volume"],
            y=profile["price_bin_mid"],
            orientation="h",
            name="Buy volume proxy",
            marker_color="#16a34a",
        )
    )
    fig.add_trace(
        go.Bar(
            x=-profile["vp_sell_volume"],
            y=profile["price_bin_mid"],
            orientation="h",
            name="Sell volume proxy",
            marker_color="#dc2626",
        )
    )
    poc_price = float(profile["vp_poc_price"].iloc[0])
    value_area = profile[profile["vp_value_area"]]
    fig.add_hline(y=poc_price, line_dash="dash", line_color="#111827", annotation_text="POC")
    if not value_area.empty:
        fig.add_hrect(
            y0=float(value_area["price_bin_mid"].min()),
            y1=float(value_area["price_bin_mid"].max()),
            fillcolor="#fde68a",
            opacity=0.20,
            line_width=0,
            annotation_text="value area",
        )
    fig.update_layout(
        title=f"{symbol} Discrete Price Bucket Volume Profile Proxy",
        barmode="relative",
        height=700,
        xaxis_title="Volume proxy (+buy / -sell)",
        yaxis_title="Price bucket",
    )
    fig.show()


## 9. Market Maker Setup Detection


In [ ]:
def detect_setups_for_symbol(symbol_frame: pd.DataFrame) -> tuple[list[dict], pd.DataFrame]:
    frame = symbol_frame.sort_values("date").reset_index(drop=True).copy()
    if len(frame) > MARKET_STRUCTURE_LOOKBACK_BARS:
        frame = frame.tail(MARKET_STRUCTURE_LOOKBACK_BARS).reset_index(drop=True)

    pivot_high_indices = np.flatnonzero(frame["pivot_high"].fillna(False).to_numpy())
    pivot_low_indices = np.flatnonzero(frame["pivot_low"].fillna(False).to_numpy())
    setups = []
    frame["mm_long_entry"] = False
    frame["mm_structure_exit"] = False
    frame["mm_target_exit"] = False
    frame["mm_micro_sweep_entry"] = False
    frame["mm_setup_id"] = np.nan
    frame["mm_stop_level"] = np.nan
    frame["mm_target_level"] = np.nan
    last_entry_idx = -1
    setup_id = 0

    for global_dip_idx in pivot_low_indices:
        if global_dip_idx <= last_entry_idx:
            continue
        previous_high_candidates = pivot_high_indices[pivot_high_indices < global_dip_idx]
        if previous_high_candidates.size == 0:
            continue
        previous_high_idx = int(previous_high_candidates[-1])
        previous_high_price = float(frame.loc[previous_high_idx, "high"])
        global_dip_price = float(frame.loc[global_dip_idx, "low"])

        update_candidates = [
            int(idx)
            for idx in pivot_high_indices
            if idx > global_dip_idx and float(frame.loc[idx, "high"]) > previous_high_price
        ]
        if not update_candidates:
            continue
        update_high_idx = update_candidates[0]
        update_high_price = float(frame.loc[update_high_idx, "high"])
        update_window = frame.loc[global_dip_idx:update_high_idx]
        update_volume_ratio = float(update_window["volume_ratio"].max())
        update_volume_confirmed = update_volume_ratio >= VOLUME_UPDATE_RATIO

        higher_low_candidates = [
            int(idx)
            for idx in pivot_low_indices
            if idx > update_high_idx and float(frame.loc[idx, "low"]) > global_dip_price
        ]
        if not higher_low_candidates:
            continue
        higher_low_idx = higher_low_candidates[0]
        higher_low_price = float(frame.loc[higher_low_idx, "low"])
        absorption_window = frame.loc[max(0, higher_low_idx - 2) : min(len(frame) - 1, higher_low_idx + 2)]
        absorption_volume_ratio = float(absorption_window["volume_ratio"].max())
        absorption_wick_ratio = float(absorption_window["lower_wick_ratio"].max())
        absorption_confirmed = (
            absorption_volume_ratio >= ABSORPTION_VOLUME_RATIO
            or absorption_wick_ratio >= ABSORPTION_LOWER_WICK_RATIO
        )

        setup_profile = setup_volume_profile_confirmation(
            frame.loc[global_dip_idx:higher_low_idx],
            global_dip_price=global_dip_price,
            higher_low_price=higher_low_price,
        )
        volume_profile_confirmed = bool(setup_profile["volume_profile_confirmed"])
        liquidity_confirmed = absorption_confirmed or volume_profile_confirmed
        if REQUIRE_LIQUIDITY_CONFIRMATION and not liquidity_confirmed:
            continue

        entry_start = min(len(frame) - 1, higher_low_idx + PIVOT_RIGHT)
        entry_end = min(len(frame), entry_start + ENTRY_LOOKAHEAD_BARS)
        entry_slice = frame.iloc[entry_start:entry_end].copy()
        momentum_entry = entry_slice[
            (entry_slice["close"] > entry_slice["high"].shift(1))
            & (entry_slice["volume_ratio"] >= VOLUME_ENTRY_RATIO)
            & (entry_slice["close"] > higher_low_price)
        ]
        micro_sweep_entry = entry_slice[
            (entry_slice["low"] < global_dip_price)
            & ((global_dip_price - entry_slice["low"]) / global_dip_price).between(MICRO_SWEEP_MIN_PCT, MICRO_SWEEP_MAX_PCT)
            & (entry_slice["close"] > global_dip_price)
            & (entry_slice["volume_ratio"] >= VOLUME_ENTRY_RATIO)
        ]

        entry_type = None
        if not micro_sweep_entry.empty:
            entry_idx = int(micro_sweep_entry.index[0])
            entry_type = "micro_sweep_reclaim"
        elif not momentum_entry.empty:
            entry_idx = int(momentum_entry.index[0])
            entry_type = "higher_low_momentum"
        else:
            continue

        if not update_volume_confirmed:
            continue

        entry_price = float(frame.loc[entry_idx, "close"])
        atr_at_entry = float(frame.loc[entry_idx, "atr"]) if pd.notna(frame.loc[entry_idx, "atr"]) else 0.0
        stop_level = min(
            global_dip_price * (1.0 - STOP_CLOSE_BUFFER_PCT),
            global_dip_price - (atr_at_entry * STOP_ATR_BUFFER),
        )
        risk_per_share = max(entry_price - stop_level, entry_price * 0.005)
        target_level = entry_price + risk_per_share * TAKE_PROFIT_R_MULTIPLIER
        target_level = max(target_level, update_high_price)

        future = frame.loc[entry_idx + 1 :]
        stop_hits = future[future["close"] <= stop_level]
        target_hits = future[future["high"] >= target_level]
        first_stop_idx = int(stop_hits.index[0]) if not stop_hits.empty else None
        first_target_idx = int(target_hits.index[0]) if not target_hits.empty else None
        exit_idx = None
        exit_reason = "open"
        if first_stop_idx is not None and (first_target_idx is None or first_stop_idx < first_target_idx):
            exit_idx = first_stop_idx
            exit_reason = "structure_close_below_stop"
            frame.loc[exit_idx, "mm_structure_exit"] = True
        elif first_target_idx is not None:
            exit_idx = first_target_idx
            exit_reason = "target_hit"
            frame.loc[exit_idx, "mm_target_exit"] = True

        setup_id += 1
        frame.loc[entry_idx, "mm_long_entry"] = True
        frame.loc[entry_idx, "mm_micro_sweep_entry"] = entry_type == "micro_sweep_reclaim"
        frame.loc[entry_idx, "mm_setup_id"] = setup_id
        frame.loc[entry_idx, "mm_stop_level"] = stop_level
        frame.loc[entry_idx, "mm_target_level"] = target_level
        if exit_idx is not None:
            frame.loc[entry_idx:exit_idx, "mm_stop_level"] = stop_level
            frame.loc[entry_idx:exit_idx, "mm_target_level"] = target_level
        else:
            frame.loc[entry_idx:, "mm_stop_level"] = stop_level
            frame.loc[entry_idx:, "mm_target_level"] = target_level

        setups.append(
            {
                "symbol": frame.loc[0, "symbol"],
                "setup_id": setup_id,
                "entry_type": entry_type,
                "previous_high_time": frame.loc[previous_high_idx, "date"],
                "previous_high": previous_high_price,
                "global_dip_time": frame.loc[global_dip_idx, "date"],
                "global_dip": global_dip_price,
                "update_high_time": frame.loc[update_high_idx, "date"],
                "update_high": update_high_price,
                "higher_low_time": frame.loc[higher_low_idx, "date"],
                "higher_low": higher_low_price,
                "entry_time": frame.loc[entry_idx, "date"],
                "entry_price": entry_price,
                "stop_level": stop_level,
                "target_level": target_level,
                "exit_time": None if exit_idx is None else frame.loc[exit_idx, "date"],
                "exit_reason": exit_reason,
                "update_volume_ratio": update_volume_ratio,
                "absorption_volume_ratio": absorption_volume_ratio,
                "absorption_wick_ratio": absorption_wick_ratio,
                "absorption_confirmed": absorption_confirmed,
                "volume_profile_confirmed": volume_profile_confirmed,
                "liquidity_confirmed": liquidity_confirmed,
                "profile_poc_price": setup_profile["profile_poc_price"],
                "profile_value_area_low": setup_profile["profile_value_area_low"],
                "profile_value_area_high": setup_profile["profile_value_area_high"],
                "cost_zone_volume_share": setup_profile["cost_zone_volume_share"],
                "cost_zone_delta_ratio": setup_profile["cost_zone_delta_ratio"],
                "weekly_macro_update": bool(frame.loc[0, "weekly_macro_update"]) if "weekly_macro_update" in frame else None,
            }
        )
        last_entry_idx = exit_idx if exit_idx is not None else entry_idx

    return setups, frame

def detect_setups_task(item):
    _, symbol_frame = item
    return detect_setups_for_symbol(symbol_frame)

setup_groups = [(symbol, symbol_frame.copy()) for symbol, symbol_frame in intraday.groupby("symbol", observed=True)]
setup_results = run_parallel_tasks(setup_groups, detect_setups_task, "market_maker_setup_detection")
all_setups = []
annotated_frames = []
for setups, annotated in setup_results:
    all_setups.extend(setups)
    annotated_frames.append(annotated)

setups_report = pd.DataFrame(all_setups)
annotated_intraday = pd.concat(annotated_frames, ignore_index=True).sort_values(["symbol", "date"]).reset_index(drop=True)

if setups_report.empty:
    print("No market-maker structure setups found with current thresholds.")
else:
    display(setups_report.sort_values("entry_time").tail(30))

setup_counts = setups_report.groupby("symbol", observed=True).size().rename("setup_count").reset_index() if not setups_report.empty else pd.DataFrame(columns=["symbol", "setup_count"])
display(setup_counts.sort_values("setup_count", ascending=False))
if not setups_report.empty:
    setup_quality = setups_report.assign(
        reward_to_risk=(setups_report["target_level"] - setups_report["entry_price"]) / (setups_report["entry_price"] - setups_report["stop_level"]),
    )
    display(setup_quality.sort_values("entry_time").tail(30))
    px.bar(
        setup_counts.sort_values("setup_count", ascending=False).head(VISUALIZATION_TOP_N),
        x="symbol",
        y="setup_count",
        title="Market Maker Setup Count By Symbol",
    ).show()
    px.scatter(
        setup_quality,
        x="update_volume_ratio",
        y="cost_zone_delta_ratio",
        size="cost_zone_volume_share",
        color="entry_type",
        hover_name="symbol",
        title="Setup Quality: Update Volume vs Cost-Zone Delta",
    ).show()
    px.histogram(
        setup_quality,
        x="reward_to_risk",
        color="entry_type",
        nbins=40,
        title="Setup Reward/Risk Distribution",
    ).show()
log_step(
    "market_maker_setups_ready",
    setup_count=len(setups_report),
    symbols_with_setups=setups_report["symbol"].nunique() if not setups_report.empty else 0,
    micro_sweep_entries=int(setups_report["entry_type"].eq("micro_sweep_reclaim").sum()) if not setups_report.empty else 0,
    volume_profile_confirmed=int(setups_report["volume_profile_confirmed"].sum()) if not setups_report.empty else 0,
    liquidity_confirmation_required=REQUIRE_LIQUIDITY_CONFIRMATION,
)


## 10. Setup Visualization


In [ ]:
plot_symbols_with_setups = (
    setups_report.groupby("symbol", observed=True).size().sort_values(ascending=False).head(PLOT_SYMBOL_LIMIT).index.to_list()
    if not setups_report.empty
    else PLOT_SYMBOLS
)
PLOT_SYMBOLS = plot_symbols_with_setups[:PLOT_SYMBOL_LIMIT]
log_step("mm_plot_selection", plotted_symbols=PLOT_SYMBOLS)

for symbol in PLOT_SYMBOLS:
    sample = annotated_intraday[annotated_intraday["symbol"] == symbol].tail(700).copy()
    if sample.empty:
        continue
    symbol_setups = setups_report[setups_report["symbol"] == symbol] if not setups_report.empty else pd.DataFrame()
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.03)
    fig.add_trace(
        go.Candlestick(
            x=sample["date"],
            open=sample["open"],
            high=sample["high"],
            low=sample["low"],
            close=sample["close"],
            name=symbol,
        ),
        row=1,
        col=1,
    )
    pivot_highs = sample[sample["pivot_high"]]
    pivot_lows = sample[sample["pivot_low"]]
    fig.add_trace(go.Scatter(x=pivot_highs["date"], y=pivot_highs["high"], mode="markers", marker=dict(symbol="triangle-down", color="#ef4444", size=8), name="Pivot High"), row=1, col=1)
    fig.add_trace(go.Scatter(x=pivot_lows["date"], y=pivot_lows["low"], mode="markers", marker=dict(symbol="triangle-up", color="#22c55e", size=8), name="Pivot Low"), row=1, col=1)
    entries = sample[sample["mm_long_entry"]]
    fig.add_trace(go.Scatter(x=entries["date"], y=entries["close"], mode="markers", marker=dict(symbol="star", color="#16a34a", size=15), name="MM Long Entry"), row=1, col=1)
    targets = sample[sample["mm_target_exit"]]
    stops = sample[sample["mm_structure_exit"]]
    fig.add_trace(go.Scatter(x=targets["date"], y=targets["high"], mode="markers", marker=dict(symbol="diamond", color="#2563eb", size=11), name="Target Exit"), row=1, col=1)
    fig.add_trace(go.Scatter(x=stops["date"], y=stops["close"], mode="markers", marker=dict(symbol="x", color="#dc2626", size=11), name="Structure Stop"), row=1, col=1)

    for _, setup in symbol_setups.tail(5).iterrows():
        fig.add_hline(y=setup["global_dip"], line_dash="dot", line_color="#7c2d12", annotation_text="global dip", row=1, col=1)
        fig.add_hline(y=setup["stop_level"], line_dash="dash", line_color="#dc2626", annotation_text="close stop", row=1, col=1)
        fig.add_hline(y=setup["target_level"], line_dash="dash", line_color="#2563eb", annotation_text="target", row=1, col=1)
        if pd.notna(setup.get("profile_poc_price", np.nan)):
            fig.add_hline(y=setup["profile_poc_price"], line_dash="dashdot", line_color="#111827", annotation_text="setup POC", row=1, col=1)
        if pd.notna(setup.get("profile_value_area_low", np.nan)) and pd.notna(setup.get("profile_value_area_high", np.nan)):
            fig.add_hrect(
                y0=setup["profile_value_area_low"],
                y1=setup["profile_value_area_high"],
                fillcolor="#fde68a",
                opacity=0.12,
                line_width=0,
                row=1,
                col=1,
            )

    bar_colors = np.where(sample["vp_delta_ratio"].fillna(0) >= 0, "#16a34a", "#dc2626")
    fig.add_trace(go.Bar(x=sample["date"], y=sample["volume_ratio"], name="Volume Ratio", marker_color=bar_colors), row=2, col=1)
    fig.add_hline(y=VOLUME_UPDATE_RATIO, line_dash="dash", line_color="#f97316", annotation_text="update volume threshold", row=2, col=1)
    fig.update_layout(title=f"{symbol} Market Maker Structure + Volume Profile Proxy", xaxis_rangeslider_visible=False, height=820)
    fig.show()


## 11. Backtest


In [ ]:
all_dates = pd.DatetimeIndex(sorted(normalize_datetime_utc_naive(annotated_intraday["date"].dropna()).dropna().unique()), name="date")
close = annotated_intraday.pivot(index="date", columns="symbol", values="close").reindex(all_dates).ffill()
entry_signals = annotated_intraday.pivot(index="date", columns="symbol", values="mm_long_entry").reindex(all_dates).fillna(False).astype(bool)
exit_signals = (
    annotated_intraday.pivot(index="date", columns="symbol", values="mm_structure_exit").reindex(all_dates).fillna(False).astype(bool)
    | annotated_intraday.pivot(index="date", columns="symbol", values="mm_target_exit").reindex(all_dates).fillna(False).astype(bool)
)
stop_levels = annotated_intraday.pivot(index="date", columns="symbol", values="mm_stop_level").reindex(all_dates).ffill()

liquidity = (prices_pd.assign(notional=prices_pd["close"] * prices_pd["volume"]).groupby("symbol", observed=True)["notional"].mean())
weights = liquidity.reindex(close.columns).fillna(0.0)
weights = weights / weights.sum() if weights.sum() > 0 else pd.Series(1 / len(close.columns), index=close.columns)

raw_stop_distance = (close - stop_levels).where(entry_signals)
min_stop_distance = close * 0.005
stop_distance = raw_stop_distance.where(raw_stop_distance >= min_stop_distance, min_stop_distance)
risk_budget_by_symbol = pd.Series(MAX_POSITION_RISK_TL, index=close.columns) * weights
shares_by_risk = stop_distance.rdiv(risk_budget_by_symbol, axis="columns").replace([np.inf, -np.inf], np.nan).apply(np.floor)
shares_by_cash = close.rdiv(INITIAL_CAPITAL * weights, axis="columns").replace([np.inf, -np.inf], np.nan).apply(np.floor)
position_size = np.minimum(shares_by_risk, shares_by_cash).fillna(0.0).where(entry_signals, 0.0)

risk_summary = pd.DataFrame(
    {
        "weight": weights,
        "risk_budget_tl": risk_budget_by_symbol,
        "max_position_size": position_size.max(),
        "entry_count": entry_signals.sum(),
    }
)
display(risk_summary.sort_values("entry_count", ascending=False).head(VISUALIZATION_TOP_N))
px.bar(
    risk_summary.sort_values("entry_count", ascending=False).head(VISUALIZATION_TOP_N).reset_index(names="symbol"),
    x="symbol",
    y="entry_count",
    color="risk_budget_tl",
    title="Backtest Entry Count And Risk Budget By Symbol",
).show()

pf_mm = vbt.Portfolio.from_signals(
    close=close.astype(float),
    entries=entry_signals,
    exits=exit_signals,
    size=position_size.astype(float),
    size_type="amount",
    init_cash=INITIAL_CAPITAL,
    fees=FEES,
    slippage=SLIPPAGE,
    freq=VECTORBT_FREQ,
    group_by=True,
    cash_sharing=True,
    call_seq="auto",
)

log_step(
    "market_maker_backtest_ready",
    trade_count=pf_mm.trades.count(),
    final_value=round(float(pf_mm.value().iloc[-1]), 2),
    entry_events=int(entry_signals.sum().sum()),
    exit_events=int(exit_signals.sum().sum()),
)
pf_mm.stats()


## 12. Performance And Trades


In [ ]:
value_curve = pf_mm.value()
returns = value_curve.pct_change().dropna()
observed_bars_per_day = value_curve.groupby(value_curve.index.date).size().median() if len(value_curve) else np.nan
periods_per_year = 252 * observed_bars_per_day if pd.notna(observed_bars_per_day) and observed_bars_per_day > 0 else 252
drawdown = value_curve / value_curve.cummax() - 1.0
stats = {
    "final_value_tl": float(value_curve.iloc[-1]),
    "total_return": float(value_curve.iloc[-1] / value_curve.iloc[0] - 1.0) if len(value_curve) > 1 else np.nan,
    "max_drawdown": float(drawdown.min()),
    "annualized_sharpe": float(np.sqrt(periods_per_year) * returns.mean() / returns.std()) if returns.std() > 0 else np.nan,
    "trade_count": int(pf_mm.trades.count()),
}
display(pd.DataFrame([stats]))
px.line(value_curve, title="Market Maker Structure Strategy Equity Curve", labels={"value": "Portfolio Value (TL)", "index": "Date"}).show()
px.area(drawdown, title="Market Maker Strategy Drawdown", labels={"value": "Drawdown", "index": "Date"}).show()

trades = pf_mm.trades.records_readable
if len(trades):
    display(trades.tail(30))
    if "Column" in trades.columns:
        trade_counts = trades.groupby("Column", observed=True).size().rename("trades").reset_index()
        px.bar(trade_counts.sort_values("trades", ascending=False).head(VISUALIZATION_TOP_N), x="Column", y="trades", title="Trades By Symbol").show()
    if "Return" in trades.columns:
        px.histogram(trades, x="Return", nbins=60, title="Trade Return Distribution").show()
else:
    print("No trades generated with current thresholds.")


## Yöntem Özeti

Bu notebook'ta eski conventional gap sistemi yerine şu kurallı yapı var:

- Universe ve filtreler aynı: TradingView BIST scan, Yahoo ownership filtresi, OHLCV completeness kontrolü.
- Haftalık bias: son majör tepe, sonrasında oluşan dip ve rebound high çıkarılıyor.
- 5 dakikalık yapı: pivot low sonrası önceki pivot high kırılırsa CHoCH/update sayılıyor.
- Higher-low: update sonrası dip, global dipin üstünde kalmalı.
- Volume teyidi: update impulsu `volume_ratio >= VOLUME_UPDATE_RATIO` ile onaylanmalı.
- Kademe proxy: fiyat bucket'ları üzerinden volume profile hesaplanıyor; her bucket için buy/sell volume proxy, toplam volume, delta, POC ve value area çıkarılıyor.
- Absorption proxy: lower wick + volume spike veya cost-zone volume profile teyidi ile temsil ediliyor.
- Entry: micro-sweep reclaim veya higher-low sonrası momentum barı.
- Stop: wick altı değil, close bazlı structure invalidation. Yani market maker hafif süpürüp geri alırsa sistem hemen stop olmaz.
- Target: risk katsayısı ve update high üstünden hesaplanır.
- Paralel işleme: weekly macro, intraday feature, volume profile ve setup detection sembol bazında `ThreadPoolExecutor` ile paralel çalışır. Zaman sırası sembol içinde korunur.
- Görselleştirme: weekly structure, volume ratio dağılımı, pivot/volume scatter, volume profile, setup quality, risk budget, equity curve, drawdown ve trade dağılımı grafiklenir.

Bence senin önerdiğin discrete fiyat gruplama yaklaşımı bu senaryo için doğru yönde. Gerçek kademe olmadığı için `buy_volume_proxy` / `sell_volume_proxy` birebir emir akışı değildir, ama market maker maliyet bölgesi, POC ve demand/supply baskısını OHLCV verisinden yaklaşık okumak için conventional gap yaklaşımından daha anlamlı bir katman sağlar.
